# 01 — Exploratory Data Analysis

Our task is to explore the electricity and gas data and prepare it for a first fraud-classification model.

We will look at the two tables, check whether the data makes sense, and create one row per client for later modelling. The invoice table is large, so it is processed in chunks instead of loading everything at once.

Run the notebook from top to bottom. Stop at the discussion prompts, talk through them with your group, and write down what you notice. The data is read from Parquet files; the optional output is a smaller CSV with one row per client.

## Business Case

Imagine that a utility company has a small team that can inspect only some clients. We want to help them decide which clients to inspect first.

- We will start with **average precision (PR-AUC)** as our main metric.
- We will also look at **precision** and **recall**. We will use a `k`, which means the number of clients the team can inspect.
- Later, we can also use ROC-AUC, a confusion matrix, and results for different groups of clients.

Before choosing a final model, we ask: 
- How many inspections are possible?
- What does an inspection cost?
- What is the cost of missing fraud or falsely accusing a customer?
These answers help us choose a `k` and a prediction threshold later.

For now, use the charts to ask good questions. A difference between fraud and non-fraud clients does not automatically explain why fraud happens.

## Imports and constants

In the next cell we import the libraries used in this notebook and define the file locations and settings. `TEST_SIZE = 0.20` means 20% of clients will go into `df_test`. `CHUNK_SIZE` controls how many invoice rows are handled at one time. You normally do not need to change these values.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import pyarrow.parquet as pq
from IPython.display import display
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

DATA_DIR = Path("data")
PARQUET_DIR = DATA_DIR / "parquet"
CHUNK_SIZE = 250_000
SAVE_FEATURES = False
AS_OF_DATE = None  # Example: "2018-12-31"; required for a real deployment backtest.
TEST_SIZE = 0.20
RSEED = 42

required_files = [PARQUET_DIR / "client_train.parquet", PARQUET_DIR / "invoice_train.parquet"]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing required files: {missing_files}.")

print(f"Using Parquet data from: {PARQUET_DIR.resolve()}")

## Step 1 - Understand the data

`client_train.parquet` has one row per client and contains the target. `invoice_train.parquet` has many invoices for each client. Before we model, we need to understand how these tables fit together.

The relationship is **one client to many invoices**. Client columns describe the customer. Invoice columns describe readings and consumption over time. For our first model, we turn the invoices into summaries for each client, for example: number of invoices, average consumption, zero-consumption rate, and whether the invoices are electricity or gas.

Think about this with your group: why would it be a problem to split invoice rows into train and test data before aggregating them?

### Load clients and make the train/test split

This cell loads the complete client table, checks its basic shape, and then makes the two DataFrames used in the project: `df_train` and `df_test`. `stratify=client["target"]` keeps the fraud percentage similar in both DataFrames. We split clients here, before exploring patterns, so `df_test` can stay unseen until the final model evaluation.

In [ ]:
client = pd.read_parquet(PARQUET_DIR / "client_train.parquet")
client["client_id"] = client["client_id"].astype("string")
client["creation_date"] = pd.to_datetime(client["creation_date"], dayfirst=True, errors="raise")
client["target"] = client["target"].astype("int8")

client_overview = pd.DataFrame({
    "rows": [len(client)],
    "unique_clients": [client["client_id"].nunique()],
    "duplicate_rows": [client.duplicated().sum()],
    "missing_cells": [int(client.isna().sum().sum())],
    "fraud_rate": [client["target"].mean()],
})
display(client_overview)
display(client.head())
display(client.dtypes.rename("dtype").to_frame())

assert client["client_id"].is_unique, "Expected one row per client."
assert set(client["target"].unique()) <= {0, 1}, "Target must be binary."

# Stratified train-test split
df_train, df_test = train_test_split(
    client,
    test_size=TEST_SIZE,
    random_state=RSEED,
    stratify=client["target"],
)

# Keep the IDs so the later merge cell can safely rebuild each split if we run it again.
train_client_ids = df_train["client_id"].copy()
test_client_ids = df_test["client_id"].copy()

### Missing values in all client rows

This check uses the complete `client` table, not only `df_train`. A zero means that column has no missing values. This is a raw-data check: it tells us whether values are empty (`NaN`), but it does not tell us whether a value is unusual or unrealistic.

In [ ]:
# Check for missing values in the complete client data
# This happens before EDA, so we see the real state of the supplied data.
display(client.isna().sum().rename("missing_values").to_frame())

### Check the split

This shows the number of clients and the target counts in `df_train` and `df_test`. The fraud share should be very similar in both tables. It is only a check that the stratified split worked; it does not use invoice information.

In [ ]:
# Print the shape of the two DataFrames
print("Train data")
print("# df_train: {}".format(df_train.shape[0]))
print("==================")
print("Test data")
print("# df_test: {}".format(df_test.shape[0]))

display(pd.crosstab(df_train["target"], columns="train", margins=True))
display(pd.crosstab(df_test["target"], columns="test", margins=True))

## Step 2 - EDA Part

### Check for missing values

The source column names `disrict` and `counter_statue` are misspelled. We keep them as they are so that the notebook matches the supplied data.

The code below reads the complete invoice file in smaller batches. This matters because there are millions of invoices and loading all of them at once can crash a notebook kernel. The missing-value result still covers every invoice row, not a sample.

### First look at invoice rows

The full invoice table is too large to display. This cell reads a small sample so we can see the column names, example values, and data types. `counter_statue` and `counter_type` are text categories: they describe a status or meter type, not an amount that can sensibly be averaged.

In [ ]:
invoice_parquet = pq.ParquetFile(PARQUET_DIR / "invoice_train.parquet")
invoice_sample = invoice_parquet.read_row_group(0).to_pandas().head(5_000)
invoice_sample["invoice_date"] = pd.to_datetime(invoice_sample["invoice_date"], errors="raise")
display(invoice_sample.head())
display(invoice_sample.dtypes.rename("inferred_dtype").to_frame())
print(f"Sample shape: {invoice_sample.shape}")

### Create one row per client

The next cell defines a function; it does not run the full calculation yet. The function reads every invoice in small batches and builds client summaries such as invoice count, first and last invoice date, consumption totals, zero-consumption rate, electricity/gas share, and the average/longest gap between invoices. It also counts missing values across the complete invoice table.

We need this because the target is stored once per client, while the invoice table has many rows per client. A model needs one consistent row per client.

In [ ]:
CONSUMPTION_COLUMNS = [f"consommation_level_{level}" for level in range(1, 5)]

def aggregate_invoices(path, chunk_size=250_000, as_of_date=None):
    partials = []
    row_hashes = []
    missing_counts = None
    status_counts = pd.Series(dtype="int64")
    tariff_counts = pd.Series(dtype="int64")
    source_rows = 0
    total_rows = 0
    excluded_after_cutoff = 0
    cutoff = pd.Timestamp(as_of_date) if as_of_date is not None else None
    started = time.perf_counter()

    parquet_file = pq.ParquetFile(path)
    reader = parquet_file.iter_batches(batch_size=chunk_size)

    for chunk_number, batch in enumerate(reader, start=1):
        chunk = batch.to_pandas()
        chunk["client_id"] = chunk["client_id"].astype("string")
        chunk["counter_statue"] = chunk["counter_statue"].astype("string")
        chunk["invoice_date"] = pd.to_datetime(chunk["invoice_date"], errors="raise")
        source_rows += len(chunk)
        # Count missing values before any optional date filtering.
        # This is the missing-value check for the complete invoice data.
        chunk_missing = chunk.isna().sum()
        missing_counts = chunk_missing if missing_counts is None else missing_counts.add(chunk_missing, fill_value=0)
        if cutoff is not None:
            after_cutoff = chunk["invoice_date"].gt(cutoff)
            excluded_after_cutoff += int(after_cutoff.sum())
            chunk = chunk.loc[~after_cutoff].copy()
        total_rows += len(chunk)
        row_hashes.append(pd.util.hash_pandas_object(chunk, index=False).to_numpy())
        status_counts = status_counts.add(chunk["counter_statue"].value_counts(), fill_value=0)
        tariff_counts = tariff_counts.add(chunk["tarif_type"].value_counts(), fill_value=0)

        chunk["total_consumption"] = chunk[CONSUMPTION_COLUMNS].sum(axis=1)
        chunk["index_delta"] = chunk["new_index"] - chunk["old_index"]
        chunk["zero_consumption"] = chunk["total_consumption"].eq(0).astype("int8")
        chunk["is_elec"] = chunk["counter_type"].eq("ELEC").astype("int8")
        chunk["is_gaz"] = chunk["counter_type"].eq("GAZ").astype("int8")
        chunk["index_went_backwards"] = chunk["index_delta"].lt(0).astype("int8")

        grouped = chunk.groupby("client_id", observed=True).agg(
            invoice_count=("client_id", "size"),
            first_invoice=("invoice_date", "min"),
            last_invoice=("invoice_date", "max"),
            total_consumption_sum=("total_consumption", "sum"),
            total_consumption_max=("total_consumption", "max"),
            index_delta_sum=("index_delta", "sum"),
            index_delta_min=("index_delta", "min"),
            index_delta_max=("index_delta", "max"),
            months_sum=("months_number", "sum"),
            zero_consumption_count=("zero_consumption", "sum"),
            elec_invoice_count=("is_elec", "sum"),
            gaz_invoice_count=("is_gaz", "sum"),
            backwards_index_count=("index_went_backwards", "sum"),
        )
        partials.append(grouped.reset_index())
        if chunk_number % 5 == 0:
            print(f"Processed {total_rows:,} invoice rows...")

    partial = pd.concat(partials, ignore_index=True)
    client_agg = partial.groupby("client_id", as_index=False).agg(
        invoice_count=("invoice_count", "sum"),
        first_invoice=("first_invoice", "min"),
        last_invoice=("last_invoice", "max"),
        total_consumption_sum=("total_consumption_sum", "sum"),
        total_consumption_max=("total_consumption_max", "max"),
        index_delta_sum=("index_delta_sum", "sum"),
        index_delta_min=("index_delta_min", "min"),
        index_delta_max=("index_delta_max", "max"),
        months_sum=("months_sum", "sum"),
        zero_consumption_count=("zero_consumption_count", "sum"),
        elec_invoice_count=("elec_invoice_count", "sum"),
        gaz_invoice_count=("gaz_invoice_count", "sum"),
        backwards_index_count=("backwards_index_count", "sum"),
    )

    # Calculate date gaps for each client. Invoice rows are stored together
    # by client in this Parquet file, so only one client's dates need to be
    # kept in memory at a time.
    gap_rows = []
    closed_client_ids = set()
    current_client_id = None
    current_date_parts = []

    def finish_current_client():
        if current_client_id is None:
            return
        dates = np.sort(np.concatenate(current_date_parts))
        if len(dates) < 2:
            gap_rows.append({
                "client_id": current_client_id,
                "invoice_gap_count": 0,
                "mean_invoice_gap_days": 0.0,
                "max_invoice_gap_days": 0,
            })
            return
        gap_days = np.diff(dates).astype("timedelta64[D]").astype(int)
        gap_rows.append({
            "client_id": current_client_id,
            "invoice_gap_count": len(gap_days),
            "mean_invoice_gap_days": gap_days.mean(),
            "max_invoice_gap_days": gap_days.max(),
        })

    gap_reader = pq.ParquetFile(path).iter_batches(
        batch_size=chunk_size,
        columns=["client_id", "invoice_date"],
    )
    for gap_batch in gap_reader:
        gap_frame = gap_batch.to_pandas()
        gap_frame["invoice_date"] = pd.to_datetime(gap_frame["invoice_date"], errors="raise")
        for client_id, group in gap_frame.groupby("client_id", sort=False):
            client_dates = group["invoice_date"].dropna().to_numpy(dtype="datetime64[ns]")
            if current_client_id is None:
                current_client_id = client_id
                current_date_parts = [client_dates]
            elif client_id == current_client_id:
                current_date_parts.append(client_dates)
            else:
                finish_current_client()
                closed_client_ids.add(current_client_id)
                if client_id in closed_client_ids:
                    raise ValueError("Invoice rows are not grouped by client; cannot calculate gaps safely.")
                current_client_id = client_id
                current_date_parts = [client_dates]
    finish_current_client()

    gap_features = pd.DataFrame(gap_rows)
    client_agg = client_agg.merge(gap_features, on="client_id", how="left", validate="one_to_one")

    client_agg["mean_consumption"] = client_agg["total_consumption_sum"] / client_agg["invoice_count"]
    client_agg["mean_index_delta"] = client_agg["index_delta_sum"] / client_agg["invoice_count"]
    client_agg["mean_months"] = client_agg["months_sum"] / client_agg["invoice_count"]
    client_agg["zero_consumption_rate"] = client_agg["zero_consumption_count"] / client_agg["invoice_count"]
    client_agg["elec_share"] = client_agg["elec_invoice_count"] / client_agg["invoice_count"]
    client_agg["active_days"] = (client_agg["last_invoice"] - client_agg["first_invoice"]).dt.days
    client_agg["backwards_index_rate"] = client_agg["backwards_index_count"] / client_agg["invoice_count"]
    client_agg["invoice_frequency_per_year"] = np.where(
        client_agg["active_days"].gt(0),
        client_agg["invoice_count"] / (client_agg["active_days"] / 365.25),
        0,
    )

    invoice_hashes = pd.Series(np.concatenate(row_hashes), copy=False)
    duplicate_row_hashes = invoice_hashes[invoice_hashes.duplicated(keep=False)].unique()
    audit = {
        "source_invoice_rows": source_rows,
        "invoice_rows": total_rows,
        "excluded_after_cutoff": excluded_after_cutoff,
        "duplicate_row_hash_matches": int(invoice_hashes.duplicated().sum()),
        "duplicate_row_hashes": duplicate_row_hashes,
        "invoice_clients": client_agg["client_id"].nunique(),
        "missing_by_column": missing_counts.sort_values(ascending=False),
        "counter_status_counts": status_counts.sort_index().astype("int64"),
        "tariff_counts": tariff_counts.sort_index().astype("int64"),
        "elapsed_seconds": time.perf_counter() - started,
    }
    return client_agg, audit

### Run the invoice aggregation

This is the longer-running cell. It processes every invoice, prints progress, and then shows the missing-value count for the complete invoice data. It also shows the values found in `counter_statue` and `tarif_type`. A result of zero missing values is useful, but still check unusual categories and extreme values later.

In [ ]:
invoice_features, invoice_audit = aggregate_invoices(
    PARQUET_DIR / "invoice_train.parquet",
    chunk_size=CHUNK_SIZE,
    as_of_date=AS_OF_DATE,
)

print(f"Processed {invoice_audit['invoice_rows']:,} rows in {invoice_audit['elapsed_seconds']:.1f}s")
print("Missing values in the complete invoice data:")
display(invoice_features.head())
display(invoice_audit["missing_by_column"].rename("missing_values").to_frame())
display(invoice_audit["counter_status_counts"].rename("invoice_rows").to_frame())
display(invoice_audit["tariff_counts"].rename("invoice_rows").to_frame())

### Add invoice summaries to train and test

This cell joins the client-level invoice summaries onto both `df_train` and `df_test`. The checks show whether every invoice client belongs to the client table and whether any client has no invoices. The EDA below uses only `df_train`; `df_test` is prepared in the same way but not explored.

In [ ]:
unknown_invoice_clients = set(invoice_features["client_id"]) - set(client["client_id"])
clients_without_invoices = set(client["client_id"]) - set(invoice_features["client_id"])
print(f"Invoice clients absent from client table: {len(unknown_invoice_clients):,}")
print(f"Clients without invoices: {len(clients_without_invoices):,}")

# Rebuild the original client-only splits before the merge.
# This makes this cell safe to run more than once: it never tries to add the same invoice features twice.
df_train = client.loc[client["client_id"].isin(train_client_ids)].copy()
df_test = client.loc[client["client_id"].isin(test_client_ids)].copy()

# Add the invoice summaries to both DataFrames.
# The client IDs were split before EDA and stay in their own DataFrame.
df_train = df_train.merge(
    invoice_features,
    on="client_id",
    how="left",
    validate="one_to_one",
)
df_test = df_test.merge(
    invoice_features,
    on="client_id",
    how="left",
    validate="one_to_one",
)
# Calculate customer tenure after merging because creation_date comes from client data and last_invoice comes from invoice data.
for frame in [df_train, df_test]:
    frame["customer_tenure_days_at_last_invoice"] = (
        frame["last_invoice"] - frame["creation_date"]
    ).dt.days

analysis = pd.concat([df_train, df_test], ignore_index=True)

assert len(analysis) == len(client)
assert not unknown_invoice_clients
display(df_train.head())
print(f"The EDA below uses df_train with {len(df_train):,} clients. df_test with {len(df_test):,} clients stays aside for later model evaluation.")

## Step 3 - Data Exploration

We explore `df_train` from here on. This lets us make decisions without looking at the test data first.

### Skewness of the features

Some clients have much larger consumption than others. We use a log scale in the charts so that those very large values do not hide the rest of the data. If you see a useful difference, ask whether that information would really be available before an inspection takes place.

### Look at the feature distributions

This cell makes one histogram for each client-level feature using `df_train`. Blue is non-fraud and red is fraud, so we can compare the two groups directly. The values are shown on a log scale because consumption, invoice counts, and invoice gaps are very uneven. Fraud is drawn with a stronger, less transparent line because it is the much smaller group. Look for long tails, many zero values, and features where the red shape differs from the blue shape. The table underneath compares the median and mean for fraud and non-fraud clients.

In [ ]:
eda_features = [
    "invoice_count",
    "active_days",
    "invoice_frequency_per_year",
    "mean_consumption",
    "total_consumption_sum",
    "total_consumption_max",
    "zero_consumption_rate",
    "mean_months",
    "elec_share",
    "mean_invoice_gap_days",
    "max_invoice_gap_days",
    "backwards_index_rate",
    "customer_tenure_days_at_last_invoice",
]

# Labels keep charts readable without changing the real column names in df_train.
feature_labels = {
    "invoice_count": "Invoice count",
    "active_days": "Active days",
    "invoice_frequency_per_year": "Invoices per year",
    "mean_consumption": "Mean consumption",
    "total_consumption_sum": "Total consumption",
    "total_consumption_max": "Maximum consumption",
    "zero_consumption_rate": "Zero-consumption rate",
    "mean_months": "Mean billing months",
    "elec_share": "Electricity invoice share",
    "mean_invoice_gap_days": "Mean invoice gap (days)",
    "max_invoice_gap_days": "Maximum invoice gap (days)",
    "backwards_index_rate": "Backward-index rate",
    "customer_tenure_days_at_last_invoice": "Tenure at last invoice (days)",
}

# Plot distributions separately for each class. Density lets us compare shapes despite class imbalance.
plot_data = df_train[eda_features + ["target"]].copy()
plot_data["Fraud status"] = plot_data["target"].map({0: "No fraud", 1: "Fraud"})
class_palette = {"No fraud": "#33658A", "Fraud": "#D62828"}
fig, ax = plt.subplots(5, 3, figsize=(18, 20))
for count, feature in enumerate(eda_features):
    plot_frame = pd.DataFrame({
        "value": np.log1p(plot_data[feature].clip(lower=0)),
        "Fraud status": plot_data["Fraud status"],
    })
    sns.histplot(
        data=plot_frame,
        x="value",
        hue="Fraud status",
        hue_order=["No fraud", "Fraud"],
        palette=class_palette,
        stat="density",
        common_norm=False,
        element="step",
        fill=False,
        bins=40,
        linewidth=2.5,
        ax=ax[int(count / 3)][count % 3],
    )
    ax[int(count / 3)][count % 3].set(title=f"{feature_labels[feature]} (log1p)", xlabel="")
for empty_ax in ax.flat[len(eda_features):]:
    empty_ax.set_visible(False)
fig.tight_layout(pad=3)
plt.show()

summary_by_target = df_train.groupby("target")[eda_features].agg(["median", "mean"])
display(summary_by_target.T)

### Look at relationships between features

The pairplot uses a small random sample of training clients because drawing all clients would be slow and unreadable. Blue points are non-fraud; red points are fraud and deliberately less transparent so they stay visible. It helps us see whether the two groups overlap. The heatmap then shows correlations. Strongly correlated features may contain similar information, so we may not need to keep all of them in a simple first model.

In [ ]:
# Pairplot: use a sample so the plot stays readable and fast
pairplot_columns = ["invoice_count", "mean_consumption", "max_invoice_gap_days", "zero_consumption_rate", "target"]
pairplot_data = df_train[pairplot_columns].sample(n=min(3_000, len(df_train)), random_state=RSEED).copy()
pairplot_data["invoice_count"] = np.log1p(pairplot_data["invoice_count"])
pairplot_data["mean_consumption"] = np.log1p(pairplot_data["mean_consumption"].clip(lower=0))
pairplot_data["max_invoice_gap_days"] = np.log1p(pairplot_data["max_invoice_gap_days"])
pairplot_data["Fraud status"] = pairplot_data["target"].map({0: "No fraud", 1: "Fraud"})
pairplot_data = pairplot_data.rename(columns={
    "invoice_count": "Invoice count (log1p)",
    "mean_consumption": "Mean consumption (log1p)",
    "max_invoice_gap_days": "Maximum invoice gap (log1p)",
    "zero_consumption_rate": "Zero-consumption rate",
})
sns.pairplot(
    pairplot_data.drop(columns="target"),
    hue="Fraud status",
    hue_order=["No fraud", "Fraud"],
    palette=class_palette,
    plot_kws={"s": 20, "alpha": 0.75},
)
plt.show()

# Correlation between numeric client-level features
corr = df_train[eda_features + ["target"]].corr(numeric_only=True)
plt.subplots(figsize=(15, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, cmap="YlGnBu_r", mask=mask, vmax=1, vmin=-1)
plt.title("Correlation between client-level features")
plt.show()

### Check the target and remaining data-quality questions

First, this cell shows the fraud/non-fraud balance in `df_train`. Then it compares fraud rates for clients with short and long invoice histories. Finally, it collects the data-quality checks in one table: duplicates, missing values, clients without invoices, and unusual date/index values. These results tell us what needs attention before modelling.

In [ ]:
# Representation of the target variable in the training data
sns.countplot(x=df_train["target"], color="#33658A").set_title("Representation of target variable")
plt.show()

# Check whether the fraud rate changes with the amount of invoice history
df_train["invoice_count_group"] = pd.qcut(df_train["invoice_count"], q=5, duplicates="drop")
history_table = pd.crosstab(
    df_train["invoice_count_group"],
    df_train["target"],
    normalize="index",
).rename(columns={0: "not_fraud", 1: "fraud"})
display(history_table)

quality_checks = pd.Series({
    "duplicate_client_rows": int(client.duplicated().sum()),
    "duplicate_client_ids": int(client["client_id"].duplicated().sum()),
    "missing_client_cells": int(client.isna().sum().sum()),
    "missing_invoice_cells": int(invoice_audit["missing_by_column"].sum()),
    "duplicate_invoice_row_hash_matches": invoice_audit["duplicate_row_hash_matches"],
    "clients_without_invoices": len(clients_without_invoices),
    "invoice_clients_without_client_row": len(unknown_invoice_clients),
    "clients_with_backwards_index": int(analysis["backwards_index_count"].gt(0).sum()),
    "creation_after_first_invoice": int(analysis["creation_date"].gt(analysis["first_invoice"]).sum()),
    "negative_customer_tenure": int(analysis["customer_tenure_days_at_last_invoice"].lt(0).sum()),
}, name="count")
display(quality_checks.to_frame())

display(analysis[["creation_date", "first_invoice", "last_invoice"]].agg(["min", "max"]))

### Inspect the records that need a closer look

The checklist above counts potential data problems. This next cell shows examples, but does **not** delete or correct anything. A duplicate hash is only a signal that two invoice rows have the same values; we inspect it before deciding whether it is a real duplicate. A backward index is a negative change in a meter reading. The date examples help us decide whether account-creation dates are reliable enough to use as a model feature.

In [ ]:
# First, summarise what each unusual pattern means and what we will decide later.
unusual_records_summary = pd.DataFrame([
    {
        "check": "Duplicate-looking invoice rows",
        "count": invoice_audit["duplicate_row_hash_matches"],
        "meaning": "Two or more invoice rows have the same values in every checked column.",
        "next step": "Inspect the matching rows; only remove them if they are true duplicates.",
    },
    {
        "check": "Clients with a backward meter index",
        "count": int(analysis["backwards_index_count"].gt(0).sum()),
        "meaning": "At least one invoice has a meter index lower than the previous reading.",
        "next step": "Keep it as a possible signal; investigate whether it is a reset or correction.",
    },
    {
        "check": "Creation date after first invoice",
        "count": int(analysis["creation_date"].gt(analysis["first_invoice"]).sum()),
        "meaning": "The recorded account-creation date is later than the first invoice date.",
        "next step": "Do not assume creation_date is a reliable start-of-service date.",
    },
    {
        "check": "Negative customer tenure",
        "count": int(analysis["customer_tenure_days_at_last_invoice"].lt(0).sum()),
        "meaning": "The last invoice is dated before the recorded account-creation date.",
        "next step": "Do not use negative tenure as-is; handle it explicitly during modelling.",
    },
])
display(unusual_records_summary)

# Read only the rows whose hashes matched, so we can inspect the possible invoice duplicates.
duplicate_hashes = set(invoice_audit["duplicate_row_hashes"])
duplicate_invoice_examples = []
if duplicate_hashes:
    invoice_file = pq.ParquetFile(PARQUET_DIR / "invoice_train.parquet")
    for batch in invoice_file.iter_batches(batch_size=CHUNK_SIZE):
        invoice_chunk = batch.to_pandas()
        # Apply the same data types and optional date filter used in aggregate_invoices.
        invoice_chunk["client_id"] = invoice_chunk["client_id"].astype("string")
        invoice_chunk["counter_statue"] = invoice_chunk["counter_statue"].astype("string")
        invoice_chunk["invoice_date"] = pd.to_datetime(invoice_chunk["invoice_date"], errors="raise")
        if AS_OF_DATE is not None:
            invoice_chunk = invoice_chunk.loc[invoice_chunk["invoice_date"].le(pd.Timestamp(AS_OF_DATE))].copy()
        row_hash = pd.util.hash_pandas_object(invoice_chunk, index=False)
        matches = invoice_chunk.loc[row_hash.isin(duplicate_hashes)]
        if not matches.empty:
            duplicate_invoice_examples.append(matches)

if duplicate_invoice_examples:
    duplicate_invoice_examples = pd.concat(duplicate_invoice_examples, ignore_index=True)
    print("Potential duplicate invoice rows (all matching rows are shown):")
    display(duplicate_invoice_examples.sort_values(["client_id", "invoice_date"]))
else:
    print("No matching invoice rows were found to display.")

# Show the clients with the most backward readings, then a small sample of each date issue.
backwards_index_examples = (
    analysis.loc[analysis["backwards_index_count"].gt(0), [
        "client_id", "target", "backwards_index_count", "index_delta_min", "invoice_count"
    ]]
    .sort_values(["backwards_index_count", "index_delta_min"], ascending=[False, True])
    .head(10)
)
display(backwards_index_examples)

creation_after_first_invoice_examples = (
    analysis.loc[analysis["creation_date"].gt(analysis["first_invoice"]), [
        "client_id", "target", "creation_date", "first_invoice", "last_invoice",
        "customer_tenure_days_at_last_invoice"
    ]]
    .sort_values(["creation_date", "first_invoice"])
    .head(10)
)
display(creation_after_first_invoice_examples)

negative_tenure_examples = (
    analysis.loc[analysis["customer_tenure_days_at_last_invoice"].lt(0), [
        "client_id", "target", "creation_date", "last_invoice",
        "customer_tenure_days_at_last_invoice"
    ]]
    .sort_values("customer_tenure_days_at_last_invoice")
    .head(10)
)
display(negative_tenure_examples)

## Results

### What we learned

- The data contains **135,493 labelled clients** and **4,476,749 invoice rows**. Each client has invoice data, and every invoice belongs to a client in the client table.
- There are **no missing values** in either source table. This means missing-value imputation is not the first issue to solve for this baseline.
- Fraud is imbalanced: **7,566 clients (5.58%)** are labelled as fraud. A model that predicts `not fraud` for everyone would be about 94% accurate but would find no fraud, so accuracy should not be our main metric.
- Fraud clients tend to have more invoice history. The median fraud client has **41 invoices** compared with **29** for a non-fraud client, and **4,739 active days** compared with **3,286**. This is a useful signal, but it may partly reflect how long a client has been observed, so we need to check it carefully.
- The strongest simple pattern is invoice count: the fraud rate rises from **0.09%** for clients with 1–7 invoices to **9.01%** for clients with 58–380 invoices. Invoice count should be in the first model, but we should check that it would be available at the point we want to make a prediction.
- Consumption also differs. Median mean consumption is **469** for fraud clients and **393** for non-fraud clients; median maximum consumption is **2,010** versus **1,284**. These are promising features, but the distributions are right-skewed, so using `log1p` is sensible for visualisation and may help a linear baseline.
- The invoice features are related to each other: invoice count, active days, total consumption, and invoice frequency describe overlapping parts of a client’s history. For a simple linear model, we should avoid adding every highly related feature without checking the result.

### Data-quality decisions

- There are **11 duplicate-looking invoice row hashes**. This is tiny compared with 4.5 million invoices. We will inspect the displayed rows before deciding whether they are true duplicates; we will not remove them automatically.
- **1,816 clients** have at least one backward meter index. This could be a correction or meter reset, but it could also be useful behaviour. Keep `backwards_index_count` / `backwards_index_rate` as features for now.
- **8,748 clients** have a creation date later than their first invoice and **397 clients** have negative tenure at their last invoice. Therefore, `creation_date` is not always a trustworthy start-of-service date. Do not use raw creation date as a feature. If we use tenure, add a flag for invalid values and decide whether to clip or set those values to missing inside the modelling pipeline.

### Next steps for the baseline

1. Keep `df_test` untouched. Develop the model with stratified cross-validation inside `df_train`.
2. Start with a DummyClassifier, then a regularised logistic regression using the client features already created. This gives us an honest baseline that is easy to explain.
3. Report **PR-AUC** as the main ranking metric because fraud is rare. Also report recall, precision, and a confusion matrix at a business-chosen threshold or review capacity.
4. Compare the baseline with and without the questionable date-derived tenure feature. Keep the backward-index feature, but document its meaning as uncertain.
5. After choosing the approach using only `df_train`, evaluate it once on `df_test` and use the false positives and false negatives for error analysis.

## Optional: save the prepared train and test data

Leave `SAVE_FEATURES = False` while you are still changing the EDA. When you are happy with the client-level features, change it to `True` and run this cell. It writes separate train and test CSV files for the baseline-model notebook.

In [ ]:
if SAVE_FEATURES:
    output_dir = DATA_DIR / "processed"
    output_dir.mkdir(parents=True, exist_ok=True)
    train_output_path = output_dir / "train_client_features.csv"
    test_output_path = output_dir / "test_client_features.csv"
    df_train.to_csv(train_output_path, index=False)
    df_test.to_csv(test_output_path, index=False)
    print(f"Saved {len(df_train):,} train rows to {train_output_path}")
    print(f"Saved {len(df_test):,} test rows to {test_output_path}")
else:
    print("Feature saving is disabled. Set SAVE_FEATURES = True when the train and test feature tables are ready to save locally.")